# 🔬 Fase 2 — Treine, Avalie e Compare
## Fine-tuning de NorBERTo vs BERTimbau para Classificação de Texto

---
**Roteiro-Desafio-NLP · Fatec DSM 6º Semestre · 2026**  
**Grupo:** ___________________________  **Data:** ___/___/______

---

### 🎯 Objetivo desta fase
Realizar **fine-tuning** (ajuste fino) do NorBERTo-base em uma tarefa de classificação de texto em português e comparar os resultados com o BERTimbau nas mesmas condições experimentais.

### 📚 O que você vai aprender
- O que é fine-tuning e como aplicá-lo
- Como preparar datasets para classificação com transformers
- Como avaliar modelos com F1, precisão e recall
- Como comparar modelos de forma controlada e justa

### 🔗 Recursos utilizados
- Modelos: `Itau-Unibanco/NorBERTo-base` vs `neuralmind/bert-base-portuguese-cased` (BERTimbau)
- Dataset escolhido pelo grupo (uma opção abaixo):
  - `HateBR` — detecção de discurso de ódio
  - `TweetSentBR` — análise de sentimentos
  - `FakeRecogna` — detecção de fake news
  - `ASSIN2` — inferência textual

---


## 📦 Passo 1 — Instalação

In [ ]:
!pip install transformers datasets torch scikit-learn seaborn matplotlib accelerate -q
print("✅ Pronto!")


## 📂 Passo 2 — Escolha e Carregamento do Dataset
Seu grupo deve escolher UM dos datasets abaixo. Comente/descomente a opção desejada.


In [ ]:
from datasets import load_dataset

# ══ OPÇÃO A: HateBR — detecção de discurso de ódio ══
# dataset = load_dataset("JAugusto97/hatecheck-pt", split="test")

# ══ OPÇÃO B: ASSIN 2 — inferência textual ══
dataset = load_dataset("assin2", split="train")

# ══ OPÇÃO C: TweetSentBR — sentimentos (via arquivo local ou adaptação) ══
# dataset = load_dataset("csv", data_files={"train": "tweetsentbr.csv"}, split="train")

print(f"✅ Dataset carregado!")
print(f"   Registros: {len(dataset)}")
print(f"   Colunas: {dataset.column_names}")
print()
print("📋 Primeiros exemplos:")
for ex in dataset.select(range(3)):
    print(ex)
    print()


## 🛠️ Passo 3 — Pré-processamento
Vamos preparar o dataset para os dois modelos. O pré-processamento deve ser **idêntico** para uma comparação justa.


In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import torch

# ─── Configurações ───────────────────────────────────────────────
MAX_LEN = 128
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

# ─── Identifica colunas do dataset (adapte se necessário) ────────
# Para ASSIN2: texto = 'premise', rótulo = 'entailment_judgment'
COL_TEXTO = "premise"
COL_ROTULO = "entailment_judgment"

# Mapeia rótulos para inteiros
rotulos_unicos = list(set(dataset[COL_ROTULO]))
label2id = {r: i for i, r in enumerate(sorted(rotulos_unicos))}
id2label = {v: k for k, v in label2id.items()}
NUM_CLASSES = len(label2id)

print(f"📊 Classes encontradas ({NUM_CLASSES}): {label2id}")

class TextDataset(Dataset):
    def __init__(self, textos, rotulos, tokenizer):
        self.encodings = tokenizer(textos, truncation=True, padding=True,
                                   max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(rotulos)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

textos = dataset[COL_TEXTO]
rotulos_num = [label2id[r] for r in dataset[COL_ROTULO]]

# Split treino/teste (80/20)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    textos, rotulos_num, test_size=0.2, random_state=42, stratify=rotulos_num)

print(f"\n✅ Split realizado:")
print(f"   Treino: {len(X_train)} exemplos")
print(f"   Teste:  {len(X_test)} exemplos")


## 🚀 Passo 4 — Fine-tuning do NorBERTo
Vamos treinar o NorBERTo-base na tarefa de classificação. O processo adiciona uma **camada de classificação** no topo do modelo.


In [ ]:
from transformers import AutoModelForSequenceClassification, AdamW, get_scheduler
from sklearn.metrics import f1_score, classification_report
import time

def treinar_e_avaliar(model_name, X_tr, X_te, y_tr, y_te, num_classes, epochs=NUM_EPOCHS):
    """Treina e avalia um modelo de classificação de texto."""
    print(f"\n{'='*60}")
    print(f"🤖 Modelo: {model_name}")
    print(f"{'='*60}")
    
    # Tokenizador e datasets
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = TextDataset(X_tr, y_tr, tokenizer)
    test_ds  = TextDataset(X_te, y_te, tokenizer)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
    
    # Modelo com camada de classificação
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes, ignore_mismatched_sizes=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print(f"   Dispositivo: {device}")
    
    # Otimizador e scheduler
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    num_steps = len(train_dl) * epochs
    scheduler = get_scheduler("linear", optimizer, num_warmup_steps=num_steps//10,
                              num_training_steps=num_steps)
    
    # Treinamento
    inicio = time.time()
    historico = []
    for ep in range(epochs):
        model.train()
        loss_ep = 0
        for batch in train_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            loss_ep += loss.item()
        historico.append(loss_ep / len(train_dl))
        print(f"   Época {ep+1}/{epochs} — Loss: {historico[-1]:.4f}")
    
    tempo_treino = time.time() - inicio
    
    # Avaliação
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in test_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    
    f1 = f1_score(labels, preds, average="macro")
    print(f"\n📊 Resultados finais:")
    print(f"   F1-Score (macro): {f1:.4f}")
    print(f"   Tempo de treino:  {tempo_treino:.1f}s")
    print(classification_report(labels, preds, target_names=[id2label[i] for i in range(num_classes)]))
    
    return {"modelo": model_name, "f1": f1, "tempo": tempo_treino, "historico": historico}

# Fine-tuning NorBERTo
resultado_norberto = treinar_e_avaliar(
    "Itau-Unibanco/NorBERTo-base",
    X_train, X_test, y_train, y_test, NUM_CLASSES)


## ⚖️ Passo 5 — Fine-tuning do BERTimbau (baseline de comparação)

In [ ]:
# Fine-tuning BERTimbau — mesmas condições!
resultado_bertimbau = treinar_e_avaliar(
    "neuralmind/bert-base-portuguese-cased",
    X_train, X_test, y_train, y_test, NUM_CLASSES)


## 📊 Passo 6 — Comparação Visual dos Resultados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

resultados = [resultado_norberto, resultado_bertimbau]
modelos = [r["modelo"].split("/")[-1] for r in resultados]
f1s = [r["f1"] for r in resultados]
tempos = [r["tempo"] for r in resultados]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Comparação: NorBERTo vs BERTimbau", fontsize=14, fontweight='bold')

# F1-Score
cores = ["#0D9488", "#6366F1"]
axes[0].bar(modelos, f1s, color=cores, width=0.5)
axes[0].set_title("F1-Score (Macro)")
axes[0].set_ylim(0, 1)
for i, v in enumerate(f1s):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha='center', fontweight='bold')

# Tempo de treino
axes[1].bar(modelos, tempos, color=cores, width=0.5)
axes[1].set_title("Tempo de Treino (s)")
for i, v in enumerate(tempos):
    axes[1].text(i, v + 0.5, f"{v:.1f}s", ha='center', fontweight='bold')

# Curvas de loss
for i, r in enumerate(resultados):
    axes[2].plot(range(1, len(r["historico"])+1), r["historico"],
                 marker='o', label=modelos[i], color=cores[i])
axes[2].set_title("Loss por Época (Treino)")
axes[2].set_xlabel("Época")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("comparacao_fase2.png", dpi=150, bbox_inches='tight')
plt.show()

# Tabela resumo
print("\n" + "="*50)
print("TABELA RESUMO — FASE 2")
print("="*50)
print(f"{'Modelo':<30} {'F1':>8} {'Tempo':>10}")
print("-"*50)
for r in resultados:
    print(f"{r['modelo'].split('/')[-1]:<30} {r['f1']:>8.4f} {r['tempo']:>9.1f}s")
print("="*50)
vencedor = max(resultados, key=lambda x: x['f1'])
print(f"\n🏆 Melhor F1: {vencedor['modelo'].split('/')[-1]} ({vencedor['f1']:.4f})")


## 📝 Avaliação — Fase 2

**Q1.** O que é fine-tuning em modelos de linguagem pré-treinados?  
( ) Treinar o modelo do zero com dados novos  
( ) Ajustar os pesos de um modelo pré-treinado para uma tarefa específica  
( ) Remover camadas do modelo para deixá-lo mais rápido  
( ) Converter o modelo para outro formato

**Q2.** Qual camada é adicionada ao NorBERTo para classificação de sequências?  
( ) Uma camada de atenção adicional  
( ) Uma camada linear (fully connected) de classificação  
( ) Uma camada de tokenização especial  
( ) Uma camada de embedding adicional

**Q3.** Por que usar os mesmos hiperparâmetros ao comparar NorBERTo e BERTimbau?  
( ) Para economizar tempo de computação  
( ) Para garantir que as diferenças nos resultados sejam do modelo, não da configuração  
( ) Porque o Hugging Face exige isso  
( ) Para evitar overfitting nos dois modelos

**Q4.** O F1-Score macro é calculado como:  
( ) Média ponderada pelo tamanho de cada classe  
( ) Média simples do F1 de cada classe, ignorando desbalanceamento  
( ) Apenas o F1 da classe positiva  
( ) A razão entre precisão e recall global

**Q5.** O que é o learning rate scheduler linear usado no treinamento?  
( ) Mantém o learning rate constante durante todo o treino  
( ) Aumenta o learning rate progressivamente  
( ) Reduz o learning rate gradualmente ao longo dos steps de treino  
( ) Define automaticamente o melhor learning rate

**Q6.** O que indica um valor de loss decrescente ao longo das épocas?  
( ) O modelo está sofrendo overfitting  
( ) O modelo está aprendendo a tarefa progressivamente  
( ) O modelo está esquecendo o pré-treinamento  
( ) O dataset está mal balanceado

**Q7.** No contexto do artigo NorBERTo, qual foi o resultado no benchmark MRPC do PLUE?  
( ) NorBERTo-large obteve F1 de 0.8873  
( ) NorBERTo-large obteve F1 de 0.9191  
( ) BERTimbau-large obteve F1 de 0.9191  
( ) Albertina obteve o melhor resultado

**Q8.** O que é o AdamW utilizado como otimizador?  
( ) Uma variante do SGD para redes recorrentes  
( ) Adam com weight decay desacoplado, comum para fine-tuning de transformers  
( ) Um otimizador específico do PyTorch para NLP  
( ) Uma versão mais lenta porém mais precisa do Adam

**Q9.** Por que é importante fazer stratify no train_test_split?  
( ) Para acelerar o processo de divisão  
( ) Para garantir que a proporção de classes seja mantida nos dois conjuntos  
( ) Para embaralhar os dados antes da divisão  
( ) Para remover duplicatas do dataset

**Q10.** O BERTimbau foi treinado em qual corpus principal?  
( ) Aurora-PT (331 bilhões de tokens)  
( ) Wikipedia em português  
( ) brWaC (~2,68 bilhões de tokens de páginas web brasileiras)  
( ) FAQ_BACEN do Itaú-Unibanco
